**Create BigQuery Table**

In [ ]:
from google.cloud import bigquery
from google.oauth2 import service_account

# =========================================
# AUTHENTICATION
# =========================================

SERVICE_ACCOUNT_FILE = "../Keys/BigQueryKey.json"

credentials = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE
)

PROJECT_ID = "depihealthnux"
DATASET_ID = "Depihealthnux_Main"
TABLE_ID = "Rx_Medications"

TABLE_REF = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"

client = bigquery.Client(
    credentials=credentials,
    project=PROJECT_ID
)

# =========================================
# SCHEMA
# =========================================

schema = [

    bigquery.SchemaField(
        "MD_Line_Key",
        "STRING",
        mode="REQUIRED"
    ),

    bigquery.SchemaField(
        "Booking_Key",
        "STRING"
    ),

    bigquery.SchemaField(
        "Patient_U_ID",
        "STRING"
    ),

    bigquery.SchemaField(
        "Medication_Name",
        "STRING"
    ),

    bigquery.SchemaField(
        "Medictaion_Dose",
        "NUMERIC"
    ),

    bigquery.SchemaField(
        "Medictaion_Dose_Unit",
        "STRING"
    ),

    bigquery.SchemaField(
        "Frequency",
        "INTEGER"
    ),

    bigquery.SchemaField(
        "Frequency_Unit",
        "STRING"
    )

]

table = bigquery.Table(
    TABLE_REF,
    schema=schema
)

table.clustering_fields = [
    "Patient_U_ID",
    "Booking_Key"
]

client.create_table(
    table,
    exists_ok=True
)

print("=================================")
print("BigQuery table created successfully")
print(TABLE_REF)
print("=================================")

BigQuery table created successfully
depihealthnux.Depihealthnux_Main.Rx_Medications


**Create Postgres Table**

In [ ]:
import sys
import psycopg2

sys.path.append("..")
from Keys.PostGresKey import POSTGRES_URL

# =========================================
# CONNECT
# =========================================

conn = psycopg2.connect(POSTGRES_URL)
cursor = conn.cursor()

# =========================================
# CREATE SEQUENCE
# =========================================

cursor.execute("""

CREATE SEQUENCE IF NOT EXISTS medication_line_seq;

""")

# =========================================
# CREATE TABLE
# =========================================

create_table_query = """
CREATE TABLE IF NOT EXISTS Rx_Medications (

    MD_Line_Key TEXT PRIMARY KEY
    DEFAULT (
        'MDL' ||
        LPAD(
            nextval('medication_line_seq')::TEXT,
            6,
            '0'
        )
    ),

    Booking_Key TEXT,

    Patient_U_ID TEXT,

    Medication_Name TEXT,

    Medictaion_Dose NUMERIC(12,1),

    Medictaion_Dose_Unit TEXT,

    Frequency INTEGER,

    Frequency_Unit TEXT

);
"""

cursor.execute(create_table_query)

# =========================================
# INDEXES
# =========================================

cursor.execute("""

CREATE INDEX IF NOT EXISTS idx_rx_booking
ON Rx_Medications(Booking_Key);

""")

cursor.execute("""

CREATE INDEX IF NOT EXISTS idx_rx_patient
ON Rx_Medications(Patient_U_ID);

""")

conn.commit()

print("=================================")
print("PostgreSQL table created successfully")
print("Table: Rx_Medications")
print("=================================")

cursor.close()
conn.close()

PostgreSQL table created successfully
Table: Rx_Medications


**Sync BigQuery Postgres**

In [3]:
import sys
import pandas as pd
import psycopg2

from psycopg2.extras import execute_values

from google.cloud import bigquery
from google.oauth2 import service_account

sys.path.append("..")
from Keys.PostGresKey import POSTGRES_URL

# =========================================
# BIGQUERY AUTH
# =========================================

credentials = service_account.Credentials.from_service_account_file(
    "../Keys/BigQueryKey.json"
)

client = bigquery.Client(
    credentials=credentials,
    project="depihealthnux"
)

# =========================================
# READ BIGQUERY
# =========================================

query = """

SELECT

    MD_Line_Key,
    Booking_Key,
    Patient_U_ID,
    Medication_Name,
    Medictaion_Dose,
    Medictaion_Dose_Unit,
    Frequency,
    Frequency_Unit

FROM `depihealthnux.Depihealthnux_Main.Rx_Medications`

ORDER BY MD_Line_Key

"""

df = client.query(query).to_dataframe()

print(df.head(3))
print(f"Rows Retrieved: {len(df)}")

# =========================================
# CONNECT POSTGRES
# =========================================

conn = psycopg2.connect(POSTGRES_URL)
cursor = conn.cursor()

cursor.execute("""
DELETE FROM Rx_Medications;
""")

if not df.empty:

    execute_values(
        cursor,
        """
        INSERT INTO Rx_Medications (

            MD_Line_Key,
            Booking_Key,
            Patient_U_ID,
            Medication_Name,
            Medictaion_Dose,
            Medictaion_Dose_Unit,
            Frequency,
            Frequency_Unit

        )
        VALUES %s
        """,
        df.values.tolist(),
        page_size=1000
    )

# =========================================
# RESET SEQUENCE
# =========================================

cursor.execute("""

SELECT setval(
    'medication_line_seq',
    COALESCE(
        (
            SELECT MAX(
                CAST(
                    REPLACE(MD_Line_Key,'MDL','')
                    AS INTEGER
                )
            )
            FROM Rx_Medications
        ),
        1
    ),
    true
);

""")

conn.commit()

print(f"Inserted {len(df)} rows")

cursor.execute("""

SELECT *
FROM Rx_Medications
ORDER BY MD_Line_Key
LIMIT 3

""")

print("\nFirst 3 Rows From PostgreSQL")

for row in cursor.fetchall():
    print(row)

cursor.close()
conn.close()

Empty DataFrame
Columns: [MD_Line_Key, Booking_Key, Patient_U_ID, Medication_Name, Medictaion_Dose, Medictaion_Dose_Unit, Frequency, Frequency_Unit]
Index: []
Rows Retrieved: 0
Inserted 0 rows

First 3 Rows From PostgreSQL


**Sync Postgres To BigQuery**

In [4]:
import sys
import pandas as pd
import psycopg2

from google.cloud import bigquery
from google.oauth2 import service_account

sys.path.append("..")
from Keys.PostGresKey import POSTGRES_URL

# =========================================
# BIGQUERY AUTH
# =========================================

credentials = service_account.Credentials.from_service_account_file(
    "../Keys/BigQueryKey.json"
)

client = bigquery.Client(
    credentials=credentials,
    project="depihealthnux"
)

# =========================================
# READ POSTGRES
# =========================================

conn = psycopg2.connect(POSTGRES_URL)

query = """

SELECT

    MD_Line_Key,
    Booking_Key,
    Patient_U_ID,
    Medication_Name,
    Medictaion_Dose,
    Medictaion_Dose_Unit,
    Frequency,
    Frequency_Unit

FROM Rx_Medications

ORDER BY MD_Line_Key

"""

df = pd.read_sql(query, conn)

print(df.head(3))
print(f"Rows Retrieved: {len(df)}")

conn.close()

# =========================================
# LOAD TO BIGQUERY
# =========================================

job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE"
)

job = client.load_table_from_dataframe(
    df,
    "depihealthnux.Depihealthnux_Main.Rx_Medications",
    job_config=job_config
)

job.result()

print(f"Uploaded {len(df)} rows")

verify_df = client.query("""

SELECT *
FROM `depihealthnux.Depihealthnux_Main.Rx_Medications`
LIMIT 3

""").to_dataframe()

print("\nFirst 3 Rows From BigQuery")

print(verify_df)

C:\Users\Sedeek Elmasry\AppData\Local\Temp\ipykernel_22944\3190189903.py:49: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Empty DataFrame
Columns: [md_line_key, booking_key, patient_u_id, medication_name, medictaion_dose, medictaion_dose_unit, frequency, frequency_unit]
Index: []
Rows Retrieved: 0
Uploaded 0 rows

First 3 Rows From BigQuery
Empty DataFrame
Columns: [md_line_key, booking_key, patient_u_id, medication_name, medictaion_dose, medictaion_dose_unit, frequency, frequency_unit]
Index: []
